In [1]:
import pandas as pd
import glob
import cv2
import re
from ultralytics import YOLO
import easyocr

# =========================
# MODELS
# =========================
models = {
    "YOLOv5": YOLO("models/yolov5_best.pt"),
    "YOLOv8": YOLO("models/yolov8_best.pt"),
}

reader = easyocr.Reader(['en'], gpu=True)

# =========================
# DATA
# =========================
images = glob.glob("dataset/test/images/*")

# =========================
# CLEANING
# =========================
def clean(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)

    return text

def is_plate(text):
    return 6 <= len(text) <= 9

# =========================
# METRICS STORAGE
# =========================
results = []

# =========================
# MAIN LOOP
# =========================
for model_name, model in models.items():

    correct = 0
    total = 0
    char_errors = 0

    for img_path in images:

        img = cv2.imread(img_path)

        preds = model.predict(img, conf=0.25, verbose=False)

        for r in preds:
            for box in r.boxes:

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                crop = img[y1:y2, x1:x2]

                ocr_result = reader.readtext(crop)

                pred_text = ""

                if ocr_result:
                    pred_text = ocr_result[0][1]

                pred_text = clean(pred_text)

                if not is_plate(pred_text):
                    continue

                # =========================
                # GROUND TRUTH (если есть label — позже подключим)
                # =========================
                gt_text = "UNKNOWN"

                total += 1

                # placeholder accuracy (пока без GT)
                if len(pred_text) > 0:
                    correct += 1

                char_errors += abs(len(pred_text) - 8)

    accuracy = correct / max(total, 1)

    results.append({
        "model": model_name,
        "detections": total,
        "accuracy_proxy": accuracy,
        "char_error": char_errors
    })

# =========================
# OUTPUT TABLE
# =========================
df = pd.DataFrame(results)
print(df)
df.to_csv("model_comparison.csv", index=False)

ModuleNotFoundError: No module named 'pandas'

In [2]:
import sys
print(sys.executable)

C:\Users\lindo\Downloads\ALPR-RU\venv\Scripts\python.exe
